In [2]:
import torch
import pandas as pd


In [5]:
df = pd.read_csv('/content/100_Unique_QA_Dataset.csv')

In [6]:
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [ ]:
#tokenization of data

def tokenize(text):
    text = text.lower()
    text = text.replace('?', "")
    text = text.replace("'","")
    return text.split()

tokenize("'What' is my name?") #demo test

['what', 'is', 'my', 'name']

In [13]:
#vocabulary formation

vocab = {'<UNK>': 0}

def build_vocab(row):
    tokenized_question = tokenize(row['question'])
    tokenized_answer = tokenize(row['answer'])
    
    merged_tokens = tokenized_question + tokenized_answer
    
    for token in merged_tokens:
        
        if token not in vocab:
            vocab[token] = len(vocab)


df.apply(build_vocab, axis = 1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [16]:
#words to numerical indices

def text_to_indices(text, vocab):
    indexed_text = []
    
    for token in tokenize(text):
        
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
    
    return indexed_text

In [17]:
text_to_indices("What is campusX", vocab)

[1, 2, 0]

In [18]:
from torch.utils.data import Dataset, DataLoader

In [19]:
class QADataset(Dataset):
    
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab
        
        
    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, index):
        numerical_question = text_to_indices(self.df.iloc[index]['question'],self.vocab)
        numerical_answer = text_to_indices(self.df.iloc[index]['answer'], self.vocab)
        
        
        return torch.tensor(numerical_question), torch.tensor(numerical_answer)

In [20]:
dataset = QADataset(df, vocab)

In [22]:
dataloader = DataLoader(
    dataset,
    batch_size = 1,
    shuffle = True
)

 <h3>Architecture of RNN</h3>

 `Input layer with 50 neurons because our embedding vector has 50 dimensions` <br>
 `One hidden layer which has 64 neurons` <br>
 `Output neuron has 324 neurons because we have 324 words in vocab then we will get a probabiliy against every word and output will be word with maximum probability`



In [36]:

import torch.nn as nn



class SimpleRNN(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding  = nn.Embedding(vocab_size, embedding_dim = 50)
        self.rnn = nn.RNN(50, 64, batch_first=True) #this layer gives 2 outputs first is outputs of hidden states and other is final output that's why we dont' use Sequential container as it expects one output
        self.fc = nn.Linear(64, vocab_size)
        
        
    def forward(self, question):
        embedded_question = self.embedding(question)
        hidden, final = self.rnn(embedded_question)
        output = self.fc(final.squeeze(0))
        
        return output
    

In [48]:
LEARNING_RATE = 0.01
EPOCHS = 500

model = SimpleRNN(len(vocab))


criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adadelta(model.parameters(), lr = LEARNING_RATE)

In [49]:
#training loop


for EPOCH in range(EPOCHS):
    
    total_loss = 0
    
    for question, answer in dataloader:
        
        optimizer.zero_grad()
        
        output = model(question)
        
        loss = criterion(output, answer[0])
        
        loss.backward()
        
        optimizer.step()
        
        
        total_loss = total_loss + loss.item()
        
    if EPOCH % 10 == 0:
        print(f"Epoch: {EPOCH+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 520.353030
Epoch: 11, Loss: 509.737891
Epoch: 21, Loss: 499.238289
Epoch: 31, Loss: 488.569442
Epoch: 41, Loss: 477.533551
Epoch: 51, Loss: 465.987386
Epoch: 61, Loss: 453.784237
Epoch: 71, Loss: 440.870099
Epoch: 81, Loss: 427.516152
Epoch: 91, Loss: 414.231324
Epoch: 101, Loss: 401.643132
Epoch: 111, Loss: 390.066592
Epoch: 121, Loss: 379.465775
Epoch: 131, Loss: 369.732607
Epoch: 141, Loss: 360.679082
Epoch: 151, Loss: 352.057419
Epoch: 161, Loss: 343.937205
Epoch: 171, Loss: 336.245595
Epoch: 181, Loss: 328.845177
Epoch: 191, Loss: 321.848071
Epoch: 201, Loss: 315.039536
Epoch: 211, Loss: 308.502174
Epoch: 221, Loss: 302.174043
Epoch: 231, Loss: 296.049794
Epoch: 241, Loss: 290.168661
Epoch: 251, Loss: 284.438493
Epoch: 261, Loss: 278.890171
Epoch: 271, Loss: 273.519541
Epoch: 281, Loss: 268.337673
Epoch: 291, Loss: 263.245763
Epoch: 301, Loss: 258.292668
Epoch: 311, Loss: 253.459848
Epoch: 321, Loss: 248.742128
Epoch: 331, Loss: 244.149668
Epoch: 341, Loss: 239.720

In [56]:
def predict(model, question, threshold):
    numerical_question = text_to_indices(question, vocab)
    
    question_tensor = torch.tensor(numerical_question).unsqueeze(0)
    
    model(question_tensor)
    
    probs = torch.nn.functional.softmax(output, dim=1)
    
    value, index = torch.max(probs, dim=1)
    
    if value < threshold:
        print("I don't know")
    else:
        print(list(vocab.keys())[index])
    
    
    
    

In [62]:
predict(model, "What is the capital of Japan?", 0.08)

night
